# Примеры метрик `TrajectoryMetrics`

Блокнот демонстрирует метрики на синтетических треках (район Владивостока для читаемой карты).

- Ряды задаются классом **`TrajectorySeries`**: время, долгота, широта, метка (`0` — аномалия).
- Сводный расчёт: **`TrajectoryMetrics.compute_all(prediction, ground_truth)`**.
- Визуализация: **`plot_track_maps`** — две карты (эталон и разметка модели), точки по статусу.

Геометрия ($d_i$, RMSE, max, distance loss) считается в **локальной ENU** (опора — первая точка ряда, `GeospatialMetrics` + `LocalEnuConverter`).

Метрики: **precision / recall / F1**, **Detection Delay** (mean, median, p95, max), **RMSE**, **max deviation**, **distance loss** (на заполненных траекториях). Affiliation, PR-AUC и Hausdorff в этом API не используются.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import importlib
import math
import sys
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "app" / "evaluation" / "trajectory_metrics.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Не найден корень репозитория (app/evaluation/trajectory_metrics.py)")

import app.evaluation.trajectory_metrics as trajectory_metrics_mod
import app.geo.local_enu as local_enu_mod
import app.viz.track_osm as track_osm_mod

importlib.reload(trajectory_metrics_mod)
importlib.reload(local_enu_mod)
importlib.reload(track_osm_mod)

TrajectoryMetrics = trajectory_metrics_mod.TrajectoryMetrics
TrajectorySeries = trajectory_metrics_mod.TrajectorySeries
TemporalMetrics = trajectory_metrics_mod.TemporalMetrics
LocalEnuConverter = local_enu_mod.LocalEnuConverter
plot_track_maps = track_osm_mod.plot_track_maps

METRIC_TABLE_ROWS: Tuple[Tuple[str, str], ...] = (
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1"),
    ("delay_mean", "Delay mean, с"),
    ("delay_median", "Delay median, с"),
    ("delay_p95", "Delay p95, с"),
    ("delay_max", "Delay max, с"),
    ("rmse_m", "RMSE, м"),
    ("max_deviation_m", "Max deviation, м"),
    ("distance_loss_m", "Distance loss, м"),
)
METRIC_LABELS = {key: label for key, label in METRIC_TABLE_ROWS}


def format_metric_value(key: str, value: float) -> str:
    if not math.isfinite(value):
        return "—"
    if key in {"precision", "recall", "f1"} or key.startswith("delay_"):
        return f"{value:.4f}"
    return f"{value:.2f}"


def print_metrics_table(
    metrics: Dict[str, float],
    title: str = "",
    rows: Optional[Sequence[Tuple[str, str]]] = None,
    prefix: str = "  ",
) -> None:
    if title:
        print(title)
    table_rows = rows if rows is not None else METRIC_TABLE_ROWS
    for key, label in table_rows:
        if key not in metrics:
            continue
        print(f"{prefix}{label}: {format_metric_value(key, metrics[key])}")


def metric_rows(*keys: str) -> list[Tuple[str, str]]:
    return [(key, METRIC_LABELS.get(key, key)) for key in keys]


def course(count: int = 12, step: float = 0.004) -> Tuple[np.ndarray, np.ndarray]:
    """Прямой ход на северо-восток, шаг ~500 м."""
    index = np.arange(count, dtype=np.float64)
    return 43.04 + step * index, 131.87 + step * index


def make_track(
    latitude: np.ndarray,
    longitude: np.ndarray,
    labels: np.ndarray,
    predicted: np.ndarray,
) -> Tuple[TrajectorySeries, TrajectorySeries]:
    time = np.arange(len(latitude), dtype=np.float64)
    lat = np.asarray(latitude, dtype=np.float64)
    lon = np.asarray(longitude, dtype=np.float64)
    gt_labels = np.asarray(labels, dtype=np.float64)
    pred_labels = np.asarray(predicted, dtype=np.float64)
    ground_truth = TrajectorySeries(time, lon, lat, gt_labels)
    prediction = TrajectorySeries(time, lon.copy(), lat.copy(), pred_labels)
    return prediction, ground_truth


def sample_track() -> Tuple[TrajectorySeries, TrajectorySeries]:
    """21 точка: петля 8–12, задержка детекции, ложная тревога на 15."""
    count = 21
    time = np.arange(count, dtype=np.float64)
    latitude = 43.04 + 0.001 * time
    longitude = 131.87 + 0.001 * time
    labels = np.ones(count, dtype=np.float64)
    labels[8:13] = 0.0
    latitude = latitude.copy()
    longitude = longitude.copy()
    latitude[8:13] += np.array([0.02, 0.05, 0.08, 0.05, 0.02])
    longitude[8:13] += np.array([0.01, 0.04, 0.01, -0.03, -0.01])
    predicted = labels.copy()
    predicted[8:10] = 1.0
    predicted[15] = 0.0
    ground_truth = TrajectorySeries(time, longitude, latitude, labels)
    prediction = TrajectorySeries(time, longitude.copy(), latitude.copy(), predicted)
    return prediction, ground_truth


def show_case(
    title: str,
    prediction: TrajectorySeries,
    ground_truth: TrajectorySeries,
    fields: Optional[Iterable[str]] = None,
    full_table: bool = False,
    plot: bool = True,
) -> Dict[str, float]:
    metrics = TrajectoryMetrics.compute_all(prediction, ground_truth)
    if full_table:
        print_metrics_table(metrics, title=title, prefix="  ")
    else:
        if not fields:
            raise ValueError("Укажите fields или full_table=True")
        print_metrics_table(metrics, title=title, rows=metric_rows(*fields), prefix="  ")
    print()
    if plot:
        plot_track_maps(prediction, ground_truth, title=title)
    return metrics


## Полный вызов `compute_all`

In [ ]:
reference_prediction, reference_truth = sample_track()
reference_metrics = show_case(
    "Эталон: задержка детекции и ложная тревога (все метрики)",
    reference_prediction,
    reference_truth,
    full_table=True,
)

perfect_prediction = TrajectorySeries(
    reference_truth.time,
    reference_truth.lon,
    reference_truth.lat,
    reference_truth.label.copy(),
)
perfect_metrics = TrajectoryMetrics.compute_all(perfect_prediction, reference_truth)
print("Сравнение с идеальной маской (те же координаты):")
print_metrics_table(perfect_metrics, title="Идеальная разметка", prefix="  ")


## Классификация (Precision, Recall, F1)

Положительный класс — аномалия (`label == 0`). Каждая точка ряда — отдельное бинарное решение; координаты в P/R/F1 не входят.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0.0

found = labels.copy()
found[10] = 0.0
prediction, ground_truth = make_track(latitude, longitude, labels, found)
show_case(
    "Классификация: одна лишняя тревога на точке 10",
    prediction,
    ground_truth,
    ["precision", "recall", "f1"],
)

shifted = np.ones(12)
shifted[5:9] = 0.0
prediction, ground_truth = make_track(latitude, longitude, labels, shifted)
show_case(
    "Классификация: эпизод найден со сдвигом на шаг",
    prediction,
    ground_truth,
    ["precision", "recall", "f1"],
)


## Detection Delay

Для каждого истинного эпизода $S_k=\{i: y_i=0\}$ считается задержка до первого $\hat y_i=0$ внутри эпизода. Если эпизод не обнаружен, в агрегаты попадает штраф, равный длительности эпизода. Ключи: `delay_mean`, `delay_median`, `delay_p95`, `delay_max`.


In [ ]:
latitude, longitude = course(14)
labels = np.ones(14)
labels[4:9] = 0.0
early = np.ones(14)
early[4:7] = 0.0
late = np.ones(14)
late[6:9] = 0.0
delay_fields = ("precision", "f1", "delay_mean", "delay_median", "delay_p95", "delay_max")

prediction, ground_truth = make_track(latitude, longitude, labels, early)
show_case("Delay: детекция с первой точки эпизода", prediction, ground_truth, delay_fields)
prediction, ground_truth = make_track(latitude, longitude, labels, late)
show_case("Delay: те же три точки, но с опозданием", prediction, ground_truth, delay_fields)

latitude, longitude = course(18, step=0.0035)
labels = np.ones(18)
labels[2:5] = 0.0
labels[8:11] = 0.0
labels[14:17] = 0.0
only_first = np.ones(18)
only_first[2] = 0.0
prediction, ground_truth = make_track(latitude, longitude, labels, only_first)
show_case(
    "Delay: мгновенно найдено одно событие из трёх",
    prediction,
    ground_truth,
    delay_fields,
)

for delay, start, end, detected in TemporalMetrics.per_event_delays(prediction, ground_truth):
    status = "обнаружен" if detected else "штраф (не обнаружен)"
    print(f"  эпизод [{start}:{end}], delay={delay:.2f} с, {status}")


## Геопространственные метрики

Координаты в аномальных точках заполняются линейной интерполяцией между штатными соседями (и для GT, и для prediction). Заполненные широта/долгота переводятся в ENU (опора — первая точка GT); попарные $d_i$ — евклидовы расстояния в метрах, затем:

- **RMSE** — только по точкам с $d_i > 0$;
- **max deviation** — $\max_i d_i$;
- **distance loss** — $|L_{\mathrm{truth}} - L_{\mathrm{pred}}|$ по длинам заполненных ломаных.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0.0
missed = np.ones(12)
geo_fields = ("f1", "rmse_m", "max_deviation_m", "distance_loss_m")

small = latitude.copy()
small[4:8] += 0.002
prediction, ground_truth = make_track(small, longitude, labels, missed)
show_case("Гео: пропущенный небольшой сдвиг", prediction, ground_truth, geo_fields)

loop = latitude.copy()
loop[4:8] += [0.015, 0.045, 0.07, 0.02]
prediction, ground_truth = make_track(loop, longitude, labels, missed)
show_case("Гео: пропущена петля", prediction, ground_truth, geo_fields)

prediction, ground_truth = make_track(loop, longitude, labels, labels)
show_case("Гео: петля помечена аномалией (заполнение по хорде)", prediction, ground_truth, geo_fields)

spike = latitude.copy()
spike[4:8] += [0.004, 0.004, 0.004, 0.055]
prediction, ground_truth = make_track(spike, longitude, labels, missed)
show_case(
    "Гео: max deviation ловит одну далёкую точку",
    prediction,
    ground_truth,
    ("rmse_m", "max_deviation_m"),
    plot=False,
)
